# Lesson 3.3 · Hyperparameter search

A hyperparameter (the depth of a tree, the strength of the regularization, the number of neighbors) is not learned from the data: it is
chosen by trying. `GridSearchCV` and `RandomizedSearchCV` automate the trial with cross-validation. This lesson shows
how to use them on a whole pipeline, how to read the results and why the score of the search is optimistic.

**What you will learn**

- The `step__parameter` syntax to search inside a pipeline.
- To read `cv_results_` instead of trusting only `best_params_`.
- Why the final score is measured on data that the search has never seen.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy.stats import loguniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
num = ["pclass", "age", "siblings_spouses", "parents_children", "fare"]
cat = ["sex", "port", "deck", "title"]
prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), cat),
], sparse_threshold=0)
pipe = make_pipeline(prep, LogisticRegression(max_iter=2000))
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## GridSearchCV on a pipeline

The name of a parameter inside a pipeline is `stepname__parameter` (at several levels: `columntransformer__num__simpleimputer__strategy`).
`make_pipeline` generates the names in lowercase from the type of the object. With `refit=True` (default) the search retrains the
best one on the whole training set: `best_estimator_`.

In [ ]:
grid = {
    "logisticregression__C": [0.01, 0.1, 1, 10],
    "columntransformer__num__simpleimputer__strategy": ["mean", "median"],
}
search = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc").fit(X_tr, y_tr)
print("best parameters:", search.best_params_)
print("best_score_ (cross-validation, on the training set):", round(search.best_score_, 3))
print("AUC on the test set (never seen by the search)    :", round(roc_auc_score(y_te, search.predict_proba(X_te)[:, 1]), 3))

## Reading `cv_results_`

`best_params_` gives a winner; `cv_results_` tells the story of the race. Look at it as a table: small differences compared to the standard
deviation across folds are not real differences.

In [ ]:
r = pd.DataFrame(search.cv_results_)
r[["param_logisticregression__C", "param_columntransformer__num__simpleimputer__strategy",
   "mean_test_score", "std_test_score", "rank_test_score"]].sort_values("rank_test_score").head(6).round(4)

## Random search

With many hyperparameters the grid explodes: 5 values for 6 parameters are 15,625 combinations. `RandomizedSearchCV` samples a
fixed number (`n_iter`) from distributions, much more efficient when only a few parameters really matter. For
parameters that vary by orders of magnitude you sample on a logarithmic scale (`loguniform`).

In [ ]:
pipe_lr = make_pipeline(prep, LogisticRegression(max_iter=2000))
search_c = RandomizedSearchCV(pipe_lr, {"logisticregression__C": loguniform(1e-3, 1e2)}, n_iter=12, cv=cv, scoring="roc_auc", random_state=0).fit(X_tr, y_tr)
tried = search_c.cv_results_["param_logisticregression__C"].astype(float)
print("values of C tried:", np.sort(tried).round(4))
print("best C:", round(search_c.best_params_["logisticregression__C"], 4), "| best_score_", round(search_c.best_score_, 3))

The 12 values of `C` are spread over several orders of magnitude (from 0.08 to 66), a few in every decade: that is what `loguniform` does. A uniform distribution between 0 and 100 would put nine values out of ten above 10
and would almost never try the small values, where the best one is here (0.44). With several hyperparameters, `RandomizedSearchCV` does the same for each of them.

In [ ]:
pipe_rf = make_pipeline(prep, RandomForestClassifier(random_state=0))
distributions = {"randomforestclassifier__n_estimators": [100, 200, 400],
                 "randomforestclassifier__max_depth": [3, 5, 8, None],
                 "randomforestclassifier__min_samples_leaf": [1, 3, 5, 10]}
rnd = RandomizedSearchCV(pipe_rf, distributions, n_iter=12, cv=cv, scoring="roc_auc", random_state=0).fit(X_tr, y_tr)
print("best:", rnd.best_params_)
print("best_score_ ", round(rnd.best_score_, 3), "| test", round(roc_auc_score(y_te, rnd.predict_proba(X_te)[:, 1]), 3))

## Why `best_score_` is optimistic

`best_score_` is the **maximum** among many noisy estimates on the same folds: even if all the configurations were equivalent, the
best one would have a score above the mean by pure luck. The more configurations you try, the more inflated the estimate. The number to
report is the one on the test set (or a nested cross-validation), touched **once**, after the search.
In this example the test set is even a bit higher than `best_score_`: with 328 rows the estimate has an uncertainty of about
±0.03, so the optimism shows up **on average** over many repetitions (you will measure it in exercise C3), not in every single case.

> **Common mistake.** Looking at the test set, changing something in the pipeline and looking at it again. After a couple of cycles the test set
> has become a second validation set and no longer estimates anything.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Does `refit=True` have an effect on the reported score?</summary>

No: `best_score_` is the cross-validated mean and does not change. `refit=True` retrains the best model on the whole training set (to use it with `predict`): that is the model you evaluate on the test set.

</details>

<details><summary>When do I prefer RandomizedSearchCV to GridSearchCV?</summary>

When I have more than two or three hyperparameters or continuous parameters: random search explores many more distinct values of the important parameters for the same number of trainings, because it does not repeat the same values in every combination.

</details>